In [80]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import json
from pathlib import Path

import pandas as pd


/mnt/sdd1/atharvas/formulacode/datasmith


In [81]:
rem_tasks = set(map(tuple, pd.read_csv("dataset/pending_manual_review_tasks.csv")[['repo_name', 'pr_base_sha']].itertuples(index=False)))

all_tasks = set([(path.parent.name, path.name) for path in Path("dataset/formulacode_verified").glob("**/*") if path.is_dir() if 'cache' not in str(path) and path.parents[1].name == 'formulacode_verified'])

tasks_path = Path("dataset/formulacode_verified")
tasks_path.exists(), len(rem_tasks), len(all_tasks)

(True, 48, 186)

In [87]:
def get_status(task_path: Path) -> dict:
    verification_file = task_path / "verification_success.json"
    failure_file = task_path / "failure.json"
    # read the one that was written more recently
    if verification_file.exists() and failure_file.exists():
        if verification_file.stat().st_mtime > failure_file.stat().st_mtime:
            # verification is more recent
            # remove failure file
            failure_file.unlink()
            d = json.loads(verification_file.read_text())
            d['task_path'] = str(task_path)
            d['ok'] = True
            return d
        else:
            d = json.loads(failure_file.read_text())
            d['task_path'] = str(task_path)
            d['ok'] = False
            return d
    if verification_file.exists():
        d = json.loads(verification_file.read_text())
        d['task_path'] = str(task_path)
        d['ok'] = True
        return d
    if failure_file.exists():
        d = json.loads(failure_file.read_text())
        d['task_path'] = str(task_path)
        d['ok'] = False
        return d
    raise FileNotFoundError(f"No status file found in {task_path}")

statuses = []
# for _, row in rem_tasks.iterrows():
# for (repo_name, pr_base_sha) in all_tasks:
for (repo_name, pr_base_sha) in rem_tasks:
    norm_repo = str(repo_name).replace("/", "_")
    task_path = tasks_path / norm_repo / str(pr_base_sha)
    status = get_status(task_path)
    statuses.append(status)
status_df = pd.DataFrame(statuses)
status_df['ok'].value_counts()


ok
False    37
True     11
Name: count, dtype: int64

In [83]:
status_df[~status_df['ok']]['return_code'].value_counts()


return_code
1.0    5
Name: count, dtype: int64

In [84]:
status_df[~status_df['ok']]['task_path'].values

array(['dataset/formulacode_verified/geopandas_geopandas/321dc873526949d5ab22ca6f7b4c06ef3d7d0f0a',
       'dataset/formulacode_verified/Qiskit_qiskit/e5ec41358896f2655239a4256d909a133925938c',
       'dataset/formulacode_verified/Qiskit_qiskit/cc2edc9f7172d75ec454fdaf5748326cd3720f90',
       'dataset/formulacode_verified/pandas-dev_pandas/9918c84043a340cc0a62625273c842c05f3d71b1',
       'dataset/formulacode_verified/pandas-dev_pandas/18bc585ae6eb6918911243b9486bb2c1a9dec570',
       'dataset/formulacode_verified/pandas-dev_pandas/ad98c2b771326502900e007a830ff69cb2e6c6e1',
       'dataset/formulacode_verified/Qiskit_qiskit/53667d167e2de2f841d3b781877427f0b459289b',
       'dataset/formulacode_verified/Qiskit_qiskit/6f482400e56345fe260d2144bf86acfd563855bc',
       'dataset/formulacode_verified/Qiskit_qiskit/a335504abd52f4f08183fb276c187951bc40385e',
       'dataset/formulacode_verified/pandas-dev_pandas/c8646541e9a2e27cc14e550c722364ded1dcba5f',
       'dataset/formulacode_verified/Q

In [85]:
tasks_todo = set(status_df[~status_df['ok']]['task_path'].values)


In [86]:
imp_tasks = """
dataset/formulacode_verified/pandas-dev_pandas/d77d5e54f9fb317f61caa1985df8ca156df921e1
dataset/formulacode_verified/pandas-dev_pandas/a730486036790f3cd26145542257a837e241144c
dataset/formulacode_verified/pandas-dev_pandas/6725e37684a24afeaea2757fb3512c58d37cef86
dataset/formulacode_verified/pandas-dev_pandas/1a9c8a4820f2028e868fbcb5018140ca29468e4d
dataset/formulacode_verified/pandas-dev_pandas/8f0c4d2ca6856cc345917d62c3f989ada00617c0
dataset/formulacode_verified/pandas-dev_pandas/68ef1dac651f3012e6da0ec6cab00ac8f0160ed4
dataset/formulacode_verified/pandas-dev_pandas/02e2baed7769bb62620cfa198f8e4fc302ab145b
dataset/formulacode_verified/pandas-dev_pandas/e5301a8ef999259f23c2603c79e1cc933861a40b
dataset/formulacode_verified/pandas-dev_pandas/aa4922a96b4c6c34b5fc6943437d902f6cc662b1
dataset/formulacode_verified/pandas-dev_pandas/af76d785ccae9383f5cf9b73e0bce79aa8ae8326
dataset/formulacode_verified/pandas-dev_pandas/87d3fe4702bf885ab8f9c01d22804352189469f2
dataset/formulacode_verified/pandas-dev_pandas/1d56672407d44bd825c8905d1663e277b244c056
dataset/formulacode_verified/pandas-dev_pandas/92a52e231534de236c4e878008a4365b4b1da291
dataset/formulacode_verified/pandas-dev_pandas/dc830eab5516f6911c894abdb1e3782cb18c503e
dataset/formulacode_verified/pandas-dev_pandas/031e9bbc3a9db4ab78be8b477b15d7173a1f625e
dataset/formulacode_verified/pandas-dev_pandas/7012d6a60724bf55d1fc0d03d6c50484aa19fb85
dataset/formulacode_verified/pandas-dev_pandas/1f622e2b5303650fa5e497e4552d0554e51049cb
dataset/formulacode_verified/pandas-dev_pandas/457690995ccbfc5b8eee80a0818d62070d078bcf
dataset/formulacode_verified/pandas-dev_pandas/984d75543fce5c68f0dcc4f0b3500256e4a9c0c9
dataset/formulacode_verified/pandas-dev_pandas/0cedcbf6ec769fbc6075c3be1bed9087d85e2dec
dataset/formulacode_verified/pandas-dev_pandas/0e11d6dfde943dd3c355aba178eb732c6e6d6223
dataset/formulacode_verified/pandas-dev_pandas/72814750548b5ead5b08cd9b90d56a23c9a520ea
dataset/formulacode_verified/pandas-dev_pandas/746e5eee860b6e143c33c9b985e095dac2e42677
dataset/formulacode_verified/pandas-dev_pandas/b89f1d0d05f4c9f360985abc6bda421d73bae85f
dataset/formulacode_verified/pandas-dev_pandas/2b82b8635d17c3a020e6e40ba72b9cc6a76f3149
dataset/formulacode_verified/pandas-dev_pandas/b2d9ec17c52084ee2b629633c9119c01ea11d387
dataset/formulacode_verified/pandas-dev_pandas/10b60441d1760ddad562206716a9ed4fb24c66a6
dataset/formulacode_verified/pandas-dev_pandas/0e12bfcd4b4d4e6f5ebe6cc6f99c0a5836df4478
dataset/formulacode_verified/pandas-dev_pandas/562523602b4ac10d1b30d813d7c2bfe02adc0469
dataset/formulacode_verified/pandas-dev_pandas/fcb8b809e91e7df47f3c20c457e9f6cb58c2f067
dataset/formulacode_verified/pandas-dev_pandas/34f39a9b053703e39f58ad1d1edc559c2465cf78
dataset/formulacode_verified/pandas-dev_pandas/5789f15402a97bbaf590c8de2696ef94c22a6bf9
dataset/formulacode_verified/pandas-dev_pandas/7bdf69f45f6ee6fae3d137ea2ab5651082c83c25
dataset/formulacode_verified/pandas-dev_pandas/1625d5a9112a0143c1dc59eadab825536e7e1240
dataset/formulacode_verified/pandas-dev_pandas/7c876ed63294a260f3a7a3e3071127a7d2e2659e
dataset/formulacode_verified/pandas-dev_pandas/38e29ab7d049015b428a880ed0a160b10418fae6
dataset/formulacode_verified/pandas-dev_pandas/5955ca6645e45d23c978076ab8e556cb91ef124c
dataset/formulacode_verified/pandas-dev_pandas/fe07fd5a201c551df679053fbe068302a1337a4a
dataset/formulacode_verified/pandas-dev_pandas/7e88122a2a829bc982b6471948c21cd1096ed980
dataset/formulacode_verified/pandas-dev_pandas/3c15cfdf0e961c4e8f74a205bac6d34e0930f988
dataset/formulacode_verified/pandas-dev_pandas/c7fa61113b4cf09581e8f31f6053f2e64b83a9fc
dataset/formulacode_verified/pandas-dev_pandas/5fc2ed2703a1370207f4ebad834e665b6c2ad42f
dataset/formulacode_verified/pandas-dev_pandas/3c96b8ff6d399fbec8d4d533e8e8618c592bb64b
dataset/formulacode_verified/pandas-dev_pandas/c8646541e9a2e27cc14e550c722364ded1dcba5f
""".strip().split("\n")

# Which of these tasks are present in tasks_todo?
[print("python dataset/verify.py --task", t, "& \\") for t in set(imp_tasks).intersection(tasks_todo)]

python dataset/verify.py --task dataset/formulacode_verified/pandas-dev_pandas/c8646541e9a2e27cc14e550c722364ded1dcba5f & \
python dataset/verify.py --task dataset/formulacode_verified/pandas-dev_pandas/92a52e231534de236c4e878008a4365b4b1da291 & \


[None, None]

In [79]:
!ls dataset/formulacode_verified/uxarray_uxarray/e88b1da5257c0ae4d74b3fd5cbb16bb58215ab27

ls: cannot access 'dataset/formulacode_verified/uxarray_uxarray/e88b1da5257c0ae4d74b3fd5cbb16bb58215ab27': No such file or directory
